# 🔩 Steel Defect Classification (Binary) with the NEU Surface Defect Database

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samtzai/course_ai_for_industrial_control_students/blob/main/notebooks/03_steel_defects_classification/n12_steel_defect_neu_binary.ipynb)

> **En Colab:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU** (el notebook también funciona en CPU, solo tarda más).

In this notebook we train a small **Convolutional Neural Network (CNN)** in PyTorch on **real images of hot-rolled steel strips**: the
[NEU Surface Defect Database](http://faculty.neu.edu.cn/songkc/en/zdylm/263270/list/index.htm) (K. Song & Y. Yan, Northeastern University).
It contains **1,800 grayscale images of 200×200 px**, 300 per defect type:

| Class | What it looks like | Typical origin |
|---|---|---|
| `crazing` (Cr) | network of fine surface cracks | thermal stress, cooling |
| `inclusion` (In) | dark spots/streaks of non-metallic particles | slag / refractory particles trapped in the steel |
| `patches` (Pa) | irregular dark areas | scale, contamination |
| `pitted_surface` (PS) | small pits / craters | corrosion, rolled-in particles |
| `rolled-in_scale` (RS) | oxide scale pressed into the surface | descaling problems before rolling |
| `scratches` (Sc) | straight lines in any direction | contact with rolls or guides |

📚 Theory: CNN basics, convolution and pooling (Parte 3, slides 21–46); confusion matrix, threshold and metrics (Parte 2, slides 56–69).

## 🏭 The binary task: is it a *critical* defect?

NEU has **no "defect-free" class** (every image contains a defect), so "defect vs. no defect" is not possible with this dataset.
A realistic binary decision on an inspection line is instead **"critical defect → stop / reject the coil segment"** vs **"other defect → continue and log it"**.

By default we use **`inclusion` as the critical class**: non-metallic inclusions are internal discontinuities that act as **crack-initiation points** in fatigue and during forming (deep drawing, bending),
so a strip with inclusions is usually downgraded or rejected. The other five classes are mostly surface defects that can often be accepted, re-classified or removed by grinding.
You can change `CRITICAL_CLASS` below (e.g. `'crazing'`, cracks) and compare.

Consequences for the model:
- The problem is **imbalanced** (1 critical image for every 5 non-critical) → we use `pos_weight` in the loss.
- **Missing a critical defect (FN)** is much worse than a false alarm (FP) → we look at **recall** and choose the **decision threshold** on the validation set.

In [ ]:
IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

import importlib.util
import subprocess
import sys

missing = [package for package in ('kagglehub', 'gdown') if importlib.util.find_spec(package) is None]
if missing:
    if importlib.util.find_spec('pip') is None:
        subprocess.run([sys.executable, '-m', 'ensurepip'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
print('Running in Colab' if IN_COLAB else 'Running locally')


In [ ]:
# 📦 Import Required Libraries
import os
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, precision_recall_curve, average_precision_score)

# Reproducibility: same split and same initial weights every time
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')

# ⏱️ QUICK_RUN=1 (environment variable) only shortens training for automatic tests. Keep it False in class.
QUICK_RUN = os.environ.get('QUICK_RUN', '0') == '1'

# 🔧 Main parameters (feel free to change them and report what happens)
IMG_SIZE = 128            # images are resized from 200x200 to 128x128 to train faster
CRITICAL_CLASS = 'inclusion'
EPOCHS = 4 if QUICK_RUN else 25
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
print('QUICK_RUN =', QUICK_RUN, '| EPOCHS =', EPOCHS)

In [ ]:
# 📥 Download the NEU Surface Defect Database into ./data
# Sources tried in order (no Kaggle login needed for the first two):
#   1) kagglehub: 'kaustubhdikshit/neu-surface-defect-database' (public Kaggle mirror of NEU-DET)
#   2) Official Google Drive link of NEU-DET published by the dataset authors (via gdown)
#   3) Manual: download NEU-DET (or NEU-CLS) and unzip it anywhere inside ./data
import os
import zipfile
from pathlib import Path
from PIL import Image

DATA_DIR = Path('./data')
DATA_DIR.mkdir(exist_ok=True)
os.environ.setdefault('KAGGLEHUB_CACHE', str((DATA_DIR / 'kagglehub').resolve()))  # keep everything in ./data

NEU_CLASSES = ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
NEU_KAGGLE_HANDLE = 'kaustubhdikshit/neu-surface-defect-database'
NEU_GDRIVE_ID = '1qrdZlaDi272eA79b0uCwwqPrm2Q_WI3k'   # NEU-DET.zip, official link from the authors' page


def list_neu_images(root):
    """All NEU images below `root`. Class and index come from the file name: <class>_<index>.jpg"""
    files = {}
    for p in Path(root).rglob('*.jpg'):
        cls, _, idx = p.stem.rpartition('_')
        if cls in NEU_CLASSES and idx.isdigit():
            files.setdefault(p.name, p)  # de-duplicate by file name
    return sorted(files.values())


def download_neu():
    if len(list_neu_images(DATA_DIR)) >= 1800:
        return DATA_DIR
    errors = []
    try:
        import kagglehub
        path = kagglehub.dataset_download(NEU_KAGGLE_HANDLE)
        if len(list_neu_images(path)) >= 1800:
            return Path(path)
        errors.append(f'kagglehub: fewer than 1800 NEU images found in {path}')
    except Exception as e:  # noqa: BLE001
        errors.append(f'kagglehub: {e}')
    try:
        import gdown
        zip_path = DATA_DIR / 'NEU-DET.zip'
        if not zip_path.exists():
            gdown.download(id=NEU_GDRIVE_ID, output=str(zip_path), quiet=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(DATA_DIR)
        if len(list_neu_images(DATA_DIR)) >= 1800:
            return DATA_DIR
        errors.append('Google Drive: fewer than 1800 NEU images found after extraction')
    except Exception as e:  # noqa: BLE001
        errors.append(f'Google Drive: {e}')
    raise RuntimeError(
        'Could not download NEU automatically.\n' + '\n'.join(errors) +
        '\nManual option: download NEU-DET from https://www.kaggle.com/datasets/' + NEU_KAGGLE_HANDLE +
        ' or from http://faculty.neu.edu.cn/songkc/en/zdylm/263270/list/index.htm and unzip it inside ./data')


NEU_ROOT = download_neu()
print(f'NEU images found: {len(list_neu_images(NEU_ROOT))} in {NEU_ROOT}')

In [ ]:
# 🧱 Load the images (grayscale, resized) and build a reproducible train / validation / test split
def load_neu(root, img_size):
    paths = list_neu_images(root)
    X = np.stack([np.array(Image.open(p).convert('L').resize((img_size, img_size), Image.BILINEAR))
                  for p in paths])                                       # (N, H, W) uint8
    y = np.array([NEU_CLASSES.index(p.stem.rpartition('_')[0]) for p in paths])
    idx = np.array([int(p.stem.rpartition('_')[2]) for p in paths])     # image number inside its class
    return X, y, idx


def blocked_split(y, idx, block_size=10, fractions=(0.6, 0.2, 0.2), seed=42):
    """Stratified split by blocks of consecutive image numbers.
    NEU images are crops of the same steel strips and neighbouring numbers can be very similar,
    so we keep each block of `block_size` consecutive images in ONE split (avoids optimistic test results)."""
    rng = np.random.default_rng(seed)
    parts = {'train': [], 'val': [], 'test': []}
    for c in np.unique(y):
        members = np.where(y == c)[0]
        blocks = (idx[members] - 1) // block_size
        order = rng.permutation(np.unique(blocks))
        n_tr = int(round(fractions[0] * len(order)))
        n_va = int(round(fractions[1] * len(order)))
        for name, sel in [('train', order[:n_tr]), ('val', order[n_tr:n_tr + n_va]), ('test', order[n_tr + n_va:])]:
            parts[name].extend(members[np.isin(blocks, sel)])
    return tuple(np.array(sorted(parts[k])) for k in ('train', 'val', 'test'))


X_all, y_all, idx_all = load_neu(NEU_ROOT, IMG_SIZE)
train_idx, val_idx, test_idx = blocked_split(y_all, idx_all, seed=SEED)
print('All images:', X_all.shape, '| classes:', NEU_CLASSES)
for name, ids in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    print(f'{name:>5}: {len(ids):4d} images | per class: {np.bincount(y_all[ids], minlength=6).tolist()}')

In [ ]:
# 👀 Look at the data: 4 examples of each defect type
fig, axes = plt.subplots(4, 6, figsize=(10, 7))
for c, name in enumerate(NEU_CLASSES):
    examples = train_idx[y_all[train_idx] == c][:4]
    for r, i in enumerate(examples):
        axes[r, c].imshow(X_all[i], cmap='gray', vmin=0, vmax=255)
        axes[r, c].axis('off')
    axes[0, c].set_title(name + (' ⚠️' if name == CRITICAL_CLASS else ''), fontsize=9,
                         color='red' if name == CRITICAL_CLASS else 'black')
plt.suptitle('NEU surface defects (red = critical class for the binary task)')
plt.tight_layout()
plt.show()

## 🧪 Prepare the binary dataset

- Label **1 = critical defect** (`CRITICAL_CLASS`), **0 = other defect**.
- Pixel values are scaled to [0, 1] and standardised with the **mean/std of the training set only** (no information from validation/test).
- Tensors have shape `(N, 1, H, W)`: PyTorch uses *channels first* and our images have a single (gray) channel.

In [ ]:
critical_id = NEU_CLASSES.index(CRITICAL_CLASS)
y_bin = (y_all == critical_id).astype(np.float32)

X_float = X_all.astype(np.float32) / 255.0
MEAN, STD = X_float[train_idx].mean(), X_float[train_idx].std()
X_norm = (X_float - MEAN) / STD
print(f'Train mean/std used for normalisation: {MEAN:.3f} / {STD:.3f}')


def make_loader(ids, shuffle):
    ds = TensorDataset(torch.from_numpy(X_norm[ids]).unsqueeze(1), torch.from_numpy(y_bin[ids]))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)


torch.manual_seed(SEED)
train_loader = make_loader(train_idx, shuffle=True)
val_loader = make_loader(val_idx, shuffle=False)
test_loader = make_loader(test_idx, shuffle=False)

for name, ids in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    n_pos = int(y_bin[ids].sum())
    print(f'{name:>5}: {len(ids)} images | critical={n_pos} | other={len(ids) - n_pos}')

## 🧠 Build the CNN

Four blocks **Conv → BatchNorm → ReLU → MaxPool** (128 → 64 → 32 → 16 → 8 px) followed by **Global Average Pooling** (Parte 3, slide 43) and a linear layer with **one output (a logit)**.

- We do **not** put a sigmoid in the model: `BCEWithLogitsLoss` applies it internally in a numerically stable way.
- `pos_weight = n_negative / n_positive` makes every critical example count ~5 times more in the loss, compensating the imbalance.

In [ ]:
def conv_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, kernel_size=3, padding=1), nn.BatchNorm2d(c_out),
                         nn.ReLU(), nn.MaxPool2d(2))


class SteelDefectCNN(nn.Module):
    def __init__(self, num_outputs=1, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, 64), conv_block(64, 128))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(128, num_outputs))

    def forward(self, x):
        return self.classifier(self.gap(self.features(x)))


torch.manual_seed(SEED)
model = SteelDefectCNN(num_outputs=1).to(device)
n_pos = y_bin[train_idx].sum()
pos_weight = torch.tensor([(len(train_idx) - n_pos) / n_pos], device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'pos_weight = {pos_weight.item():.2f}')

## 🏋️ Train the model

Classic PyTorch loop: `zero_grad → forward → loss → backward → step`. After every epoch we measure loss and accuracy on the **validation** set and keep the weights of the epoch with the **lowest validation loss** (a simple form of early stopping).

In [ ]:
def run_epoch(model, loader, train):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb).squeeze(1)
            loss = criterion(logits, yb)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(yb)
            correct += ((logits > 0).float() == yb).sum().item()
            n += len(yb)
    return total_loss / n, correct / n


history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
best_val, best_state, best_epoch = np.inf, None, 0
start = time.time()
for epoch in range(EPOCHS):
    tr_loss, tr_acc = run_epoch(model, train_loader, train=True)
    va_loss, va_acc = run_epoch(model, val_loader, train=False)
    for k, v in zip(history, [tr_loss, tr_acc, va_loss, va_acc]):
        history[k].append(v)
    if va_loss < best_val:
        best_val, best_state, best_epoch = va_loss, copy.deepcopy(model.state_dict()), epoch + 1
    print(f'Epoch {epoch + 1:2d}/{EPOCHS} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | val loss {va_loss:.4f} acc {va_acc:.3f}')
print(f'Training time: {time.time() - start:.1f} s | best epoch (min val loss): {best_epoch}')
_ = model.load_state_dict(best_state)  # restore the best epoch

In [ ]:
# 📈 Visualize Training History
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history['accuracy'], label='train'); axes[0].plot(history['val_accuracy'], label='validation')
axes[1].plot(history['loss'], label='train'); axes[1].plot(history['val_loss'], label='validation')
for ax, title in zip(axes, ['Accuracy', 'Loss (weighted BCE)']):
    ax.axvline(best_epoch - 1, color='gray', ls='--', label='best epoch')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 🎚️ Choose the decision threshold on the validation set

The network outputs a probability `p = sigmoid(logit)`. The default rule *"critical if p > 0.5"* is arbitrary (Parte 2, slides 56–67).
For a critical defect we want a **high recall**. Here we pick, **on the validation set**, the highest threshold that still gives `recall ≥ TARGET_RECALL`, and only then we look at the test set.

In [ ]:
TARGET_RECALL = 0.95


@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    probs, labels = [], []
    for xb, yb in loader:
        probs.append(torch.sigmoid(model(xb.to(device)).squeeze(1)).cpu().numpy())
        labels.append(yb.numpy())
    return np.concatenate(probs), np.concatenate(labels)


p_val, t_val = predict_proba(model, val_loader)
p_test, t_test = predict_proba(model, test_loader)

prec, rec, thr = precision_recall_curve(t_val, p_val)
ok = np.where(rec[:-1] >= TARGET_RECALL)[0]
threshold = float(thr[ok[-1]]) if len(ok) else 0.5
print(f'Chosen threshold (validation, recall >= {TARGET_RECALL}): {threshold:.3f}')

plt.figure(figsize=(5, 4))
plt.plot(rec, prec, label=f'validation (AP={average_precision_score(t_val, p_val):.3f})')
p2, r2, _ = precision_recall_curve(t_test, p_test)
plt.plot(r2, p2, label=f'test (AP={average_precision_score(t_test, p_test):.3f})')
plt.axhline(t_test.mean(), color='gray', ls=':', label='random classifier')
plt.xlabel('Recall (critical defects found)'); plt.ylabel('Precision'); plt.title('Precision-Recall curve')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 📊 Model Evaluation on the test set

We compare the default threshold 0.5 with the threshold chosen on validation.

In [ ]:
names = ['Other defect', f'Critical ({CRITICAL_CLASS})']
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, th in zip(axes, [0.5, threshold]):
    y_pred = (p_test >= th).astype(int)
    cm = confusion_matrix(t_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=names, yticklabels=names, ax=ax, cbar=False)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title(f'Test confusion matrix (threshold={th:.2f})')
    print(f'Threshold {th:.3f}: accuracy={accuracy_score(t_test, y_pred):.3f} precision={precision_score(t_test, y_pred):.3f} '
          f'recall={recall_score(t_test, y_pred):.3f} F1={f1_score(t_test, y_pred):.3f} | FN={cm[1, 0]} FP={cm[0, 1]}')
plt.tight_layout()
plt.show()

print(classification_report(t_test, (p_test >= threshold).astype(int), target_names=names, digits=3))

In [ ]:
# 🔍 Which images does the model get wrong? (test set, chosen threshold)
y_pred = (p_test >= threshold).astype(int)
wrong = np.where(y_pred != t_test)[0]
print(f'{len(wrong)} wrong test images out of {len(t_test)}')
n_show = min(10, len(wrong))
if n_show:
    fig, axes = plt.subplots(1, n_show, figsize=(2 * n_show, 2.6), squeeze=False)
    for ax, j in zip(axes[0], wrong[:n_show]):
        i = test_idx[j]
        ax.imshow(X_all[i], cmap='gray'); ax.axis('off')
        ax.set_title(f'{NEU_CLASSES[y_all[i]]}\np={p_test[j]:.2f}', fontsize=8, color='red')
    plt.suptitle('Misclassified test images (true class, predicted probability of critical)')
    plt.tight_layout()
    plt.show()

## 📝 Summary and Next Steps

### What we did
- Loaded **real steel surface images** (NEU) and built a **block-wise, stratified train / validation / test split** with a fixed seed.
- Defined an **industrial binary task** (critical defect vs. other defect) and handled the **class imbalance** with `pos_weight`.
- Trained a small CNN with BatchNorm and Global Average Pooling, keeping the epoch with the best validation loss.
- Chose the **decision threshold on validation** according to a recall target and evaluated **once** on test.

### Things to think about (you will need your own numbers for the quiz)
- How many critical defects did your model miss (FN) on test with threshold 0.5 and with the chosen threshold? How many false alarms (FP)?
- Which non-critical classes are confused with the critical one? Look at the misclassified images.
- What happens if you remove `pos_weight` or change `CRITICAL_CLASS`?

➡️ Next notebook (**n13**): classify the **six** defect types (multiclass).